# Prompt Classifier — Full Pipeline

Run this notebook from top to bottom to go from raw datasets to a final comparison of all 4 models.

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU).  
Data prep and M1 will use CPU automatically; M2/M3/M4 will use the GPU.

**Estimated time:** ~3–4 hours total (mostly M4 fine-tune).

**Sections:**
1. Setup (clone repo, mount Drive, install deps)
2. Build data (download, dedup, split)
3. EDA
4. Train M1 — TF-IDF + LR
5. Train M2 — Arctic Embed + FFN
6. Train M3 — Frozen RoBERTa + Head
7. Train M4 — Full Fine-Tuned RoBERTa (~2–3h)
8. Evaluate all models

---
## 1. Setup

In [ ]:
GITHUB_REPO = 'https://github.com/evanpaul14/prompt-classifier.git'
DRIVE_BASE  = '/content/drive/MyDrive/polygence'

import subprocess, os, sys

subprocess.run(['nvidia-smi'], check=False)

from google.colab import drive
drive.mount('/content/drive')

DRIVE = DRIVE_BASE

repo_dir = '/content/polygence'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', GITHUB_REPO, repo_dir], check=True)
else:
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)

os.chdir(repo_dir)
sys.path.insert(0, 'src')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE}/hf_cache/datasets'
os.environ['HF_HOME']           = f'{DRIVE}/hf_cache/hub'

for subdir in ['data/processed', 'data/interim', 'models', 'reports']:
    drive_path = f'{DRIVE}/{subdir}'
    os.makedirs(drive_path, exist_ok=True)
    local_path = subdir
    parent = os.path.dirname(local_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    if os.path.islink(local_path): os.remove(local_path)
    elif os.path.isdir(local_path) and not os.listdir(local_path): os.rmdir(local_path)
    if not os.path.exists(local_path): os.symlink(drive_path, local_path)

from prompt_classifier.seeds import set_all_seeds
set_all_seeds(42)

for d in ['data/processed', 'data/interim', 'models', 'reports']:
    print(d, '->', os.path.realpath(d))
print('Setup complete!')

---
## 2. Build Data

Downloads all 7 datasets, normalizes, deduplicates (exact + MinHash), and creates stratified train/val/test splits. Skipped automatically if splits already exist on Drive.

In [ ]:
import json, pandas as pd
from prompt_classifier.config import load_config
from prompt_classifier.data import loaders, unify
from prompt_classifier.data.splits import make_splits

cfg = load_config('configs/base.yaml')

import os
if os.path.exists(cfg['data']['train_path']):
    print('Splits already exist on Drive — skipping download and build.')
    print('Delete data/processed/ on Drive to force a rebuild.')
else:
    print('Downloading and building data...')
    raw = loaders.load_all(cfg)
    print(f'Raw total: {len(raw):,} rows')
    print(raw.groupby(['source', 'y']).size())

    unified = unify.build(raw, cfg)
    print(f'After dedup: {len(unified):,} rows')

    with open(cfg['data']['stats_path']) as f:
        stats = json.load(f)
    print(json.dumps({k: v for k, v in stats.items() if k != 'per_source'}, indent=2))

    train, val, test = make_splits(unified, cfg)
    with open(cfg['data']['manifest_path']) as f:
        manifest = json.load(f)
    print('Split manifest hash:', manifest['unified_hash'])
    print(manifest['counts'])

---
## 3. EDA

In [ ]:
import matplotlib.pyplot as plt

cfg = load_config('configs/base.yaml')
train_df = pd.read_parquet(cfg['data']['train_path'])
val_df   = pd.read_parquet(cfg['data']['val_path'])
test_df  = pd.read_parquet(cfg['data']['test_path'])
all_df   = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f'Total: {len(all_df):,}  |  block={(all_df.y==1).sum():,}  safe={(all_df.y==0).sum():,}')
print()
print(all_df.groupby(['source', 'y']).size().unstack(fill_value=0))

all_df['char_len'] = all_df['prompt'].str.len()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, name) in zip(axes, [(0, 'Safe'), (1, 'Block')]):
    subset = all_df[all_df.y == label]['char_len']
    ax.hist(subset.clip(upper=subset.quantile(0.99)), bins=50, edgecolor='black')
    ax.set_title(f'{name} prompt length (chars)')
    ax.set_xlabel('chars')
plt.tight_layout()
plt.show()

---
## 4. Model 1 — TF-IDF + Logistic Regression

Baseline. CPU only, ~5 min.

In [ ]:
from prompt_classifier.models.m1_tfidf_lr import run as run_m1

report_m1 = run_m1('configs/model1_tfidf_lr.yaml')
print(f"M1  F1={report_m1['f1']:.4f}  FPR={report_m1['fpr']:.4f}  FNR={report_m1['fnr']:.4f}")
print('FPR target met:', report_m1['targets_met']['fpr_ok'])
print('FNR target met:', report_m1['targets_met']['fnr_ok'])

---
## 5. Model 2 — Arctic Embed M v2.0 + FFN

Embeddings cached to Drive on first run (~20 min to encode). GPU used for encoding.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

from prompt_classifier.models.m2_arctic_ffn import run as run_m2

report_m2 = run_m2('configs/model2_arctic_ffn.yaml')
print(f"M2  F1={report_m2['f1']:.4f}  FPR={report_m2['fpr']:.4f}  FNR={report_m2['fnr']:.4f}")
print('FPR target met:', report_m2['targets_met']['fpr_ok'])
print('FNR target met:', report_m2['targets_met']['fnr_ok'])

---
## 6. Model 3 — Frozen RoBERTa + Classification Head

RoBERTa CLS embeddings cached to Drive on first run (~25 min). Head trains in minutes.

In [ ]:
from prompt_classifier.models.m3_roberta_frozen import run as run_m3

report_m3 = run_m3('configs/model3_roberta_frozen.yaml')
print(f"M3  F1={report_m3['f1']:.4f}  FPR={report_m3['fpr']:.4f}  FNR={report_m3['fnr']:.4f}")
print('FPR target met:', report_m3['targets_met']['fpr_ok'])
print('FNR target met:', report_m3['targets_met']['fnr_ok'])

---
## 7. Model 4 — Full Fine-Tuned RoBERTa

~2–3 hours on T4. Runs a quick pilot first (10% data, 1 epoch) to verify VRAM fits before committing to the full run.

Checkpoints saved to Drive each epoch — safe to resume if the runtime disconnects.

In [ ]:
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else 'no GPU')

# Pilot: 10% data, 1 epoch — confirms VRAM and Trainer config are OK
from prompt_classifier.config import load_config
from prompt_classifier.models.m4_roberta_ft import _PromptDataset, _compute_metrics
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

cfg4 = load_config('configs/model4_roberta_ft.yaml')
tr_pilot = pd.read_parquet(cfg4['data']['train_path']).sample(frac=0.10, random_state=42)
vl_pilot = pd.read_parquet(cfg4['data']['val_path']).sample(frac=0.10, random_state=42)

tokenizer = AutoTokenizer.from_pretrained(cfg4['base_model'])
pilot_model = AutoModelForSequenceClassification.from_pretrained(cfg4['base_model'], num_labels=2)
pilot_args = TrainingArguments(
    output_dir='/tmp/m4_pilot', num_train_epochs=1,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    evaluation_strategy='epoch', save_strategy='no',
    logging_steps=20, fp16=True, report_to='none',
)
Trainer(
    model=pilot_model, args=pilot_args,
    train_dataset=_PromptDataset(tr_pilot['prompt'].tolist(), tr_pilot['y'].tolist(), tokenizer),
    eval_dataset=_PromptDataset(vl_pilot['prompt'].tolist(), vl_pilot['y'].tolist(), tokenizer),
    compute_metrics=_compute_metrics,
).train()
del pilot_model  # free VRAM before full run
torch.cuda.empty_cache()
print('Pilot complete — starting full run...')

In [ ]:
from prompt_classifier.models.m4_roberta_ft import run as run_m4

report_m4 = run_m4('configs/model4_roberta_ft.yaml')
print(f"M4  F1={report_m4['f1']:.4f}  FPR={report_m4['fpr']:.4f}  FNR={report_m4['fnr']:.4f}")
print('FPR target met:', report_m4['targets_met']['fpr_ok'])
print('FNR target met:', report_m4['targets_met']['fnr_ok'])

---
## 8. Final Comparison

In [ ]:
import subprocess
result = subprocess.run([sys.executable, 'scripts/eval_all.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown

with open('reports/comparison.md') as f:
    display(Markdown(f.read()))

# Bar chart
reports = {}
for m in ['m1', 'm2', 'm3', 'm4']:
    p = f'reports/{m}.json'
    if os.path.exists(p):
        with open(p) as f:
            reports[m] = json.load(f)

models = list(reports.keys())
f1s  = [reports[m]['f1']  for m in models]
fprs = [reports[m]['fpr'] for m in models]
fnrs = [reports[m]['fnr'] for m in models]

x = np.arange(len(models))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, vals, title, target in zip(axes,
    [f1s, fprs, fnrs],
    ['F1 (higher = better)', 'FPR (target < 0.05)', 'FNR (target < 0.10)'],
    [None, 0.05, 0.10]):
    bars = ax.bar(x, vals)
    ax.set_xticks(x); ax.set_xticklabels(models)
    ax.set_title(title)
    if target:
        ax.axhline(target, color='red', linestyle='--', label=f'target={target}')
        ax.legend()
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()